In [1]:
# Recommender with TF-IDF + Cosine Similarity
# - Items: title + labels_openai (+ optional other label column)
# - Users: labels_per_person_full (expands labels by counts/weights)
# - Builds TF-IDF profiles for users and items, ranks items by cosine
# - Prints Top-N per person and exports an Excel per person (with URL + why)

import os
import re
import ast
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --------------- Config ---------------
PATH_ITEMS  = "things_to_do_all_with_local_with_labels.csv"
PATH_USERS  = "labels_per_person_full.csv"
TOP_N = 5
OUT_DIR = "recommendations tfidf"

# --------------- Helpers ---------------
def normalize(s: str) -> str:
    """Normalize text: trim, lowercase, collapse whitespace."""
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    s = re.sub(r"\s+", " ", str(s)).strip().lower()
    return s

def parse_maybe_list(x):
    """Parse list/tuple-like strings (e.g., '["a","b"]'); fallback to raw string."""
    if pd.isna(x): 
        return ""
    s = str(x).strip()
    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        try:
            arr = ast.literal_eval(s)
            if isinstance(arr, (list, tuple)):
                return " ".join(map(str, arr))
        except Exception:
            pass
    return s

def expand_counts_to_text(series_terms, series_counts):
    """Repeat terms according to counts to weight user preferences."""
    tokens = []
    for term, cnt in zip(series_terms, series_counts):
        term = normalize(term)
        try:
            n = int(round(float(cnt)))
        except Exception:
            n = 1
        tokens.extend([term] * max(n, 1))
    return " ".join(tokens)

def detect_url_col(cols):
    """Heuristic to find a URL column in the items DataFrame."""
    lower = {c.lower(): c for c in cols}
    for key in ["detail_url", "url", "link", "website", "details_url", "detailurl", "detailURL"]:
        if key in lower:
            return lower[key]
    for c in cols:
        cl = c.lower()
        if "url" in cl or "link" in cl or ("detail" in cl and "id" not in cl):
            return c
    return None

def safe_filename(name: str) -> str:
    """Create a safe filename from a person's name."""
    name = re.sub(r"[^\w\s\-\._]", "", str(name))
    name = re.sub(r"\s+", "_", name.strip())
    return name or "person"

# --------------- Load items ---------------
df_items = pd.read_csv(PATH_ITEMS)
if "title" not in df_items.columns:
    raise ValueError("No column 'title' found in items file.")

label_cols = [c for c in df_items.columns if "label" in c.lower()]
labels_openai_col = next((c for c in label_cols if c.lower()=="labels_openai"), None)
other_labels_col  = next((c for c in label_cols if c != labels_openai_col), None)

title_norm = df_items["title"].astype(str).map(normalize)
parts = [title_norm]
if labels_openai_col:
    parts.append(df_items[labels_openai_col].fillna("").apply(parse_maybe_list).map(normalize))
if other_labels_col:
    parts.append(df_items[other_labels_col].fillna("").apply(parse_maybe_list).map(normalize))

items_text = parts[0]
for p in parts[1:]:
    items_text = (items_text + " " + p).str.strip()

df_items["_text"] = items_text

# URL support
url_col = detect_url_col(df_items.columns)
item_urls = df_items[url_col].astype(str).tolist() if url_col else [""] * len(df_items)
item_titles = df_items["title"].astype(str).tolist()

# --------------- Load users ---------------
df_users = pd.read_csv(PATH_USERS)
lower = {c.lower(): c for c in df_users.columns}
person_col = lower.get("person") or lower.get("owner") or next(
    (c for c in df_users.columns if "person" in c.lower() or "owner" in c.lower()), None
)
label_col  = lower.get("label")  or lower.get("tag")   or lower.get("keyword") or next(
    (c for c in df_users.columns if "label" in c.lower() or "tag" in c.lower()), None
)
count_col  = next(
    (c for c in df_users.columns if any(k in c.lower() for k in ["count","freq","weight","score","frequency"])), 
    None
)

if not (person_col and label_col):
    raise ValueError("Need 'person' and 'label' (or synonyms) in labels_per_person_full.csv.")

if count_col is None:
    df_users["_count"] = 1
    count_col = "_count"

profiles = []
for person, sub in df_users.groupby(person_col):
    text = expand_counts_to_text(sub[label_col].astype(str).tolist(), sub[count_col].tolist())
    text = normalize(text)
    if text:
        profiles.append({"person": str(person), "text": text})

df_profiles = pd.DataFrame(profiles)
if df_profiles.empty:
    raise ValueError("No user profiles were generated. Check input labels per person.")

# --------------- TF-IDF (people + items) ---------------
combined_text = pd.concat([df_profiles["text"], df_items["_text"]], ignore_index=True)

vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    max_features=30000,
    min_df=1,
    sublinear_tf=True
)
X = vectorizer.fit_transform(combined_text)

n_people = df_profiles.shape[0]
people_mat = X[:n_people]   # (P x V)
items_mat  = X[n_people:]   # (I x V)

# --------------- Cosine(person, item) ---------------
sim = cosine_similarity(people_mat, items_mat)  # (P x I)

# --------------- Why terms (overlap with contribution) ---------------
feat = np.array(vectorizer.get_feature_names_out())
def why_terms(person_idx, item_idx, top_k=5):
    p_vec = people_mat[person_idx]
    i_vec = items_mat[item_idx]
    p_idx = set(p_vec.indices.tolist())
    i_idx = set(i_vec.indices.tolist())
    inter = np.array(sorted(p_idx & i_idx))
    if inter.size == 0:
        return ""
    contrib = (p_vec[:, inter].toarray().ravel() * i_vec[:, inter].toarray().ravel())
    order = np.argsort(-contrib)[:top_k]
    return ", ".join(feat[inter[order]])

# --------------- Print & Export per person ---------------
os.makedirs(OUT_DIR, exist_ok=True)

for pi, person in enumerate(df_profiles["person"].tolist()):
    row = sim[pi]
    top_idx = np.argsort(-row)[:TOP_N]

    print(f"\n=== Recommendations (TF-IDF) for {person} — Top {TOP_N} ===")

    rows = []
    for rank, j in enumerate(top_idx, start=1):
        title = item_titles[j]
        score = float(row[j])
        url   = item_urls[j] if j < len(item_urls) else ""
        why   = why_terms(pi, j)

        print(f"{rank:>2}. {title}  |  score={score:.4f}  |  url: {url}\n    why: {why}")

        pretty = f"{title}  |  score={score:.4f}  |  url: {url}\nwhy: {why}"
        rows.append({
            "rank": rank,
            "title": title,
            "score": round(score, 4),
            "url": url,
            "why": why,
            "pretty": pretty
        })

    out_df = pd.DataFrame(rows, columns=["rank", "title", "score", "url", "why", "pretty"])
    fname = os.path.join(OUT_DIR, f"recs_{safe_filename(person)}.xlsx")

    # Prefer xlsxwriter for hyperlinks/formatting; fallback to openpyxl if needed
    try:
        with pd.ExcelWriter(fname, engine="xlsxwriter") as writer:
            out_df.to_excel(writer, index=False, sheet_name="recommendations")
            wb = writer.book
            ws = writer.sheets["recommendations"]

            # Optional formatting
            wrap = wb.add_format({"text_wrap": True, "valign": "top"})
            num4 = wb.add_format({"num_format": "0.0000"})

            # Make URLs clickable (column D)
            for r in range(len(out_df)):
                url_val = out_df.at[r, "url"]
                if isinstance(url_val, str) and url_val.startswith(("http://", "https://")):
                    ws.write_url(r + 1, 3, url_val, string=url_val)

            # Set column widths and formats
            ws.set_column("A:A", 6)        # rank
            ws.set_column("B:B", 50)       # title
            ws.set_column("C:C", 10, num4) # score
            ws.set_column("D:D", 70)       # url
            ws.set_column("E:E", 40, wrap) # why
            ws.set_column("F:F", 90, wrap) # pretty
    except ModuleNotFoundError:
        with pd.ExcelWriter(fname, engine="openpyxl") as writer:
            out_df.to_excel(writer, index=False, sheet_name="recommendations")

    print(f"Saved ✅ {fname}  (rows: {len(out_df)})")

print("\nFiles in output folder:")
print(os.listdir(OUT_DIR))



=== Recommendations (TF-IDF) for Caro's Album — Top 5 ===
 1. A Practical Guide to Austin From a Local's Perspective  |  score=0.0856  |  url: https://theculturetrip.com/articles/a-practical-guide-to-austin-from-a-locals-perspective-2
    why: sitting, nature, scenic view, scenic, nature outdoor
 2. Austin’s Best Live Music Venues  |  score=0.0734  |  url: https://theculturetrip.com/articles/texas-culture-guide-austin-s-9-best-live-music-venues
    why: venue, indoor, music venue, lighting, music
 3. The Best Local Galleries in Dallas  |  score=0.0661  |  url: https://theculturetrip.com/articles/the-best-local-galleries-in-dallas
    why: art, indoor, nature, display, view
 4. The Most Beautiful Castles in Texas  |  score=0.0639  |  url: https://theculturetrip.com/articles/the-most-beautiful-castles-in-texas
    why: stone, cloudy, historic, hillside, steps
 5. Who’s Afraid Of Virginia Woolf?  |  score=0.0637  |  url: https://theculturetrip.com/articles/whos-afraid-of-virginia-woolf
 

In [2]:
!pip install openpyxl


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
# opción 1 (recomendada: usa el mismo intérprete del kernel)
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install gensim

# si estás en conda y prefieres conda:
# !conda install -y -c conda-forge gensim


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 3.5 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 25.2
    Uninstalling pip-25.2:
      Successfully uninstalled pip-25.2


In [4]:
# Recommender with Word Embeddings (Word2Vec) + Cosine Similarity
# - Items: title + labels_openai (+ labels si existe)
# - Users: labels_per_person_full (expande por conteo)
# - Trains a Word2Vec on your project corpus, then averages word vectors
# - Prints Top-N per person and exports an Excel per person with URL + why

import os
import re
import ast
import random
from collections import Counter

import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

# ---------------- Config ----------------
PATH_ITEMS = "things_to_do_all_with_local_with_labels.csv"
PATH_USERS = "labels_per_person_full.csv"
EMBED_DIM = 200
WINDOW = 5
MIN_COUNT = 1      # small dataset -> keep low
EPOCHS = 100
TOP_N = 5
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# ---------------- Helpers ----------------
def normalize(s):
    if s is None:
        return ""
    s = re.sub(r"[^a-zA-Z0-9\s\-&/]", " ", str(s)).lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize(s):
    return [t for t in normalize(s).split() if t]

def parse_maybe_list(x):
    """Convert '["a","b"]' -> ['a','b']; fallback to tokenizing plain string."""
    if pd.isna(x):
        return []
    s = str(x).strip()
    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        try:
            arr = ast.literal_eval(s)
            if isinstance(arr, (list, tuple)):
                return [normalize(t) for t in arr if str(t).strip()]
        except Exception:
            pass
    return tokenize(s)

def expand_counts_to_tokens(terms, counts):
    """Repeat tokens according to counts (to weight user prefs)."""
    toks = []
    for t, c in zip(terms, counts):
        t = normalize(t)
        try:
            n = int(round(float(c)))
        except Exception:
            n = 1
        toks.extend([t]*max(n,1))
    return [w for w in toks if w]

def text_to_vec(tokens, w2v):
    """Average word vectors; 0-vector if none in vocab."""
    vecs = []
    for w in tokens:
        if w in w2v.wv:
            vecs.append(w2v.wv[w])
    if not vecs:
        return np.zeros(w2v.vector_size, dtype=np.float32)
    return np.mean(vecs, axis=0)

def top_shared_tokens(user_tokens, item_tokens, k=6):
    """Simple 'why': top shared tokens by combined frequency."""
    cu = Counter(user_tokens)
    ci = Counter(item_tokens)
    common = [(w, cu[w] + ci[w]) for w in set(cu) & set(ci)]
    common.sort(key=lambda x: -x[1])
    return ", ".join([w for w,_ in common[:k]])

def detect_url_col(cols):
    """Detect a likely URL column in items dataframe."""
    lower = {c.lower(): c for c in cols}
    # prioridades comunes
    for key in ["detail_url", "url", "link", "website", "details_url", "source_url"]:
        if key in lower:
            return lower[key]
    # heurística: columnas que contengan 'url' o 'link' o 'detail' (no id)
    for c in cols:
        cl = c.lower()
        if "url" in cl or "link" in cl or ("detail" in cl and "id" not in cl):
            return c
    return None

def safe_filename(name):
    name = re.sub(r"[^\w\s\-\._]", "", str(name))
    name = re.sub(r"\s+", "_", name.strip())
    return name or "person"

# ---------------- Load items ----------------
df_items = pd.read_csv(PATH_ITEMS)
if "title" not in df_items.columns:
    raise ValueError("Missing 'title' in items file.")

label_cols = [c for c in df_items.columns if "label" in c.lower()]
labels_openai_col = next((c for c in label_cols if c.lower()=="labels_openai"), None)
other_labels_col  = next((c for c in label_cols if c != labels_openai_col), None)

items_tokens = []
for _, r in df_items.iterrows():
    toks = []
    toks += tokenize(r.get("title", ""))
    if labels_openai_col:
        toks += parse_maybe_list(r.get(labels_openai_col, ""))
    if other_labels_col:
        toks += parse_maybe_list(r.get(other_labels_col, ""))
    items_tokens.append(toks)

item_titles = df_items["title"].astype(str).tolist()

# Detect URL column (e.g., 'detail_url')
url_col = detect_url_col(df_items.columns)

# ---------------- Load users ----------------
df_users = pd.read_csv(PATH_USERS)
lower = {c.lower(): c for c in df_users.columns}
person_col = lower.get("person") or lower.get("owner") or next((c for c in df_users.columns if "person" in c.lower() or "owner" in c.lower()), None)
label_col  = lower.get("label")  or lower.get("tag")   or lower.get("keyword") or next((c for c in df_users.columns if "label" in c.lower() or "tag" in c.lower()), None)
count_col  = next((c for c in df_users.columns if any(k in c.lower() for k in ["count","freq","weight","score","frequency"])), None)

if not (person_col and label_col):
    raise ValueError("labels_per_person_full.csv must have 'person' and 'label' (or similar) columns.")

if count_col is None:
    df_users["_count"] = 1
    count_col = "_count"

user_names = []
user_tokens = []
for person, sub in df_users.groupby(person_col):
    toks = expand_counts_to_tokens(sub[label_col].astype(str).tolist(), sub[count_col].tolist())
    user_names.append(str(person))
    user_tokens.append(toks)

# ---------------- Train Word2Vec on combined corpus ----------------
corpus = user_tokens + items_tokens
w2v = Word2Vec(
    sentences=corpus,
    vector_size=EMBED_DIM,
    window=WINDOW,
    min_count=MIN_COUNT,
    workers=1,
    sg=1,            # skip-gram for small datasets suele rendir mejor
    seed=SEED
)
w2v.train(corpus, total_examples=len(corpus), epochs=EPOCHS)

# ---------------- Build vectors ----------------
user_vecs = np.vstack([text_to_vec(toks, w2v) for toks in user_tokens])  # (P x D)
item_vecs = np.vstack([text_to_vec(toks, w2v) for toks in items_tokens]) # (I x D)

def safe_cosine(A, B):
    An = np.linalg.norm(A, axis=1, keepdims=True)
    Bn = np.linalg.norm(B, axis=1, keepdims=True)
    An[An==0] = 1.0
    Bn[Bn==0] = 1.0
    return (A @ B.T) / (An @ Bn.T)

sim = safe_cosine(user_vecs, item_vecs)  # (P x I)

# ---------------- Print & Export Top-N per user ----------------
os.makedirs("recs_by_person", exist_ok=True)

for i, person in enumerate(user_names):
    scores = sim[i]
    order = np.argsort(-scores)[:TOP_N]

    print(f"\n=== Recommendations (embeddings) for {person} — Top {TOP_N} ===")

    rows = []
    for rank, j in enumerate(order, start=1):
        why = top_shared_tokens(user_tokens[i], items_tokens[j], k=6)
        title = item_titles[j]
        score = float(scores[j])
        url_val = ""
        if url_col is not None:
            val = df_items.iloc[j][url_col]
            url_val = "" if pd.isna(val) else str(val)

        # print consola (con url + why)
        print(f"{rank:>2}. {title}  |  score={score:.4f}  |  url: {url_val}\n    why: {why}")

        # fila para Excel
        pretty = f"{title}  |  score={score:.4f}  |  url: {url_val}\nwhy: {why}"
        rows.append({
            "rank": rank,
            "title": title,
            "score": round(score, 4),
            "url": url_val,
            "why": why,
            "pretty": pretty
        })

    out_df = pd.DataFrame(rows, columns=["rank", "title", "score", "url", "why", "pretty"])
    fname = f"recs_by_person/recs_{safe_filename(person)}.xlsx"

    # Requiere openpyxl para escribir .xlsx -> pip install openpyxl
    with pd.ExcelWriter(fname, engine="openpyxl") as writer:
        out_df.to_excel(writer, index=False, sheet_name="recommendations")



=== Recommendations (embeddings) for Caro's Album — Top 5 ===
 1. Take on the Texas Treehouse Zipline  |  score=0.7309  |  url: https://theculturetrip.com/articles/take-on-the-texas-treehouse-zipline
    why: trees, park
 2. A Practical Guide to Austin From a Local's Perspective  |  score=0.7084  |  url: https://theculturetrip.com/articles/a-practical-guide-to-austin-from-a-locals-perspective-2
    why: trees, nature, river, water, sitting
 3. Finding Secret Beach In Austin, Texas  |  score=0.7048  |  url: https://theculturetrip.com/articles/finding-secret-beach-in-austin-texas
    why: trees, river, beach, outdoors, dog
 4. Enjoy Engineered Beauty At AIA Sandcastle Competition  |  score=0.6983  |  url: https://theculturetrip.com/articles/enjoy-engineered-beauty-at-aia-sandcastle-competition
    why: architecture, daytime, beach
 5. 7 Relaxing, Scenic Escapes In Fort Worth  |  score=0.6820  |  url: https://theculturetrip.com/articles/7-relaxing-scenic-escapes-in-fort-worth
    why: tr